# Signature Real-vs-Forged Classifier: End-to-End Workflow
This notebook demonstrates how to train a machine learning model to classify real vs forged signatures, save the best model, and prepare it for use in a website.

## 1. Import Required Libraries
We will use PyTorch for deep learning, torchvision for image handling, scikit-learn for evaluation, and matplotlib for visualization.

In [3]:
#%pip install scikit-learn

# Import required libraries
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import joblib

## 2. Load and Prepare Dataset
We will organize the dataset into train and validation folders, and apply necessary image transformations.

In [4]:
# Prepare train/val split from scratch every time to ensure all data is used
import shutil, random

def prepare_data():
    data_dir = 'data'
    real_dir = os.path.join(data_dir, 'real signature', 'real signature')
    fake_dir = os.path.join(data_dir, 'fake signature', 'fake signature')
    train_dir = os.path.join(data_dir, 'train')
    val_dir = os.path.join(data_dir, 'val')
    # Always clear and rebuild train/val
    for d in [train_dir, val_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
    os.makedirs(train_dir)
    os.makedirs(val_dir)
    for label, src in [('real', real_dir), ('fake', fake_dir)]:
        files = [f for f in os.listdir(src) if not f.startswith('.') and os.path.isfile(os.path.join(src, f))]
        random.shuffle(files)
        split = int(0.8 * len(files))
        train_files = files[:split]
        val_files = files[split:]
        os.makedirs(os.path.join(train_dir, label))
        os.makedirs(os.path.join(val_dir, label))
        for f in train_files:
            shutil.copy(os.path.join(src, f), os.path.join(train_dir, label, f))
        for f in val_files:
            shutil.copy(os.path.join(src, f), os.path.join(val_dir, label, f))
    return train_dir, val_dir

train_dir, val_dir = prepare_data()

# Image transformations
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Datasets and loaders
batch_size = 16
train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds = datasets.ImageFolder(val_dir, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

## 3. Split Data into Training and Test Sets
The data is already split into training and validation sets in the previous step using an 80/20 split.

## 4. Build and Train Machine Learning Model
We will define a simple CNN for image classification and train it. The best model (with lowest validation loss) will be saved.

In [5]:
# Define a simple CNN
class SignatureCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 32, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# Training loop with best model saving

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SignatureCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

best_val_loss = float('inf')
best_model_path = 'best_signature_cnn.pth'
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"  Saved new best model at epoch {epoch+1}")

Epoch 1: Train Loss=0.5050, Val Loss=0.4085
  Saved new best model at epoch 1
Epoch 2: Train Loss=0.3667, Val Loss=0.3104
  Saved new best model at epoch 2
Epoch 3: Train Loss=0.2650, Val Loss=0.2715
  Saved new best model at epoch 3
Epoch 4: Train Loss=0.2024, Val Loss=0.2481
  Saved new best model at epoch 4
Epoch 5: Train Loss=0.1409, Val Loss=0.2298
  Saved new best model at epoch 5
Epoch 6: Train Loss=0.0961, Val Loss=0.2886
Epoch 7: Train Loss=0.0668, Val Loss=0.3384
Epoch 8: Train Loss=0.0329, Val Loss=0.3686
Epoch 9: Train Loss=0.0159, Val Loss=0.4520
Epoch 10: Train Loss=0.0200, Val Loss=0.4881


## 5. Evaluate Model Performance
Load the best model and evaluate it on the validation set using classification metrics.

In [6]:
# Load the best model
model = SignatureCNN().to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['real', 'fake']))
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

        real       0.88      0.91      0.89       288
        fake       0.90      0.88      0.89       288

    accuracy                           0.89       576
   macro avg       0.89      0.89      0.89       576
weighted avg       0.89      0.89      0.89       576

[[261  27]
 [ 35 253]]


## 6. Save the Best Model
The best model is already saved as `best_signature_cnn.pth` during training. You can also export it for use in a web application.

In [7]:
# Optionally, export the model for web use (TorchScript or ONNX)
# Example: Save as TorchScript
example_input = torch.randn(1, 1, 128, 256).to(device)
traced_script_module = torch.jit.trace(model, example_input)
traced_script_module.save('signature_cnn_script.pt')
print('TorchScript model saved as signature_cnn_script.pt')

TorchScript model saved as signature_cnn_script.pt


## 7. Load the Saved Model and Make Predictions
This demonstrates how to load the saved model and use it to make predictions on new signature images, as would be done in a web application.

In [8]:
# Example: Load TorchScript model and predict on a new image
def predict_signature(image_path, model_path='signature_cnn_script.pt'):
    from PIL import Image
    model = torch.jit.load(model_path, map_location=device)
    model.eval()
    img = Image.open(image_path).convert('L')
    img = transforms.Resize((128, 256))(img)
    img = transforms.ToTensor()(img)
    img = transforms.Normalize([0.5], [0.5])(img)
    img = img.unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(img)
        pred = out.argmax(1).item()
    return 'real' if pred == 0 else 'fake'

# Example usage:
# result = predict_signature('path/to/signature.jpg')
# print('Prediction:', result)

## 8. Transfer Learning with ResNet18
We will now use a pre-trained ResNet18 model with data augmentation and early stopping to improve signature classification accuracy. The best model (CNN or ResNet) will be saved based on validation performance.

In [9]:
# Data augmentation for training
from torchvision import models
from torchvision import transforms as T

train_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((128, 256)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(8),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])
val_transform = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((128, 256)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

train_ds_resnet = datasets.ImageFolder(train_dir, transform=train_transform)
val_ds_resnet = datasets.ImageFolder(val_dir, transform=val_transform)
train_loader_resnet = DataLoader(train_ds_resnet, batch_size=16, shuffle=True)
val_loader_resnet = DataLoader(val_ds_resnet, batch_size=16)

In [10]:
# Transfer learning: ResNet18
import copy
resnet = models.resnet18(weights='IMAGENET1K_V1')
for param in resnet.parameters():
    param.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet = resnet.to(device)

criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(resnet.fc.parameters(), lr=1e-3)

best_resnet_model = None
best_resnet_val_acc = 0.0
patience = 3
patience_counter = 0
num_epochs = 15

for epoch in range(num_epochs):
    resnet.train()
    for imgs, labels in train_loader_resnet:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        out = resnet(imgs)
        loss = criterion_resnet(out, labels)
        loss.backward()
        optimizer_resnet.step()
    # Validation
    resnet.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader_resnet:
            imgs, labels = imgs.to(device), labels.to(device)
            out = resnet(imgs)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    print(f"[ResNet18] Epoch {epoch+1}: Val Acc={val_acc:.4f}")
    if val_acc > best_resnet_val_acc:
        best_resnet_val_acc = val_acc
        best_resnet_model = copy.deepcopy(resnet.state_dict())
        patience_counter = 0
        print("  Saved new best ResNet18 model.")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("  Early stopping triggered.")
            break

# Save best ResNet18 model
if best_resnet_model:
    torch.save(best_resnet_model, 'best_signature_resnet18.pth')
    print('Best ResNet18 model saved as best_signature_resnet18.pth')

[ResNet18] Epoch 1: Val Acc=0.8281
  Saved new best ResNet18 model.
[ResNet18] Epoch 2: Val Acc=0.8472
  Saved new best ResNet18 model.
[ResNet18] Epoch 3: Val Acc=0.8594
  Saved new best ResNet18 model.
[ResNet18] Epoch 4: Val Acc=0.8576
[ResNet18] Epoch 5: Val Acc=0.8663
  Saved new best ResNet18 model.
[ResNet18] Epoch 6: Val Acc=0.8559
[ResNet18] Epoch 7: Val Acc=0.8524
[ResNet18] Epoch 8: Val Acc=0.8611
  Early stopping triggered.
Best ResNet18 model saved as best_signature_resnet18.pth


In [11]:
# Evaluate best ResNet18 model
resnet.load_state_dict(torch.load('best_signature_resnet18.pth', map_location=device))
resnet.eval()
all_preds_resnet, all_labels_resnet = [], []
with torch.no_grad():
    for imgs, labels in val_loader_resnet:
        imgs = imgs.to(device)
        out = resnet(imgs)
        preds = out.argmax(1).cpu().numpy()
        all_preds_resnet.extend(preds)
        all_labels_resnet.extend(labels.numpy())
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels_resnet, all_preds_resnet, target_names=['real', 'fake']))
print(confusion_matrix(all_labels_resnet, all_preds_resnet))

              precision    recall  f1-score   support

        real       0.87      0.86      0.87       288
        fake       0.86      0.88      0.87       288

    accuracy                           0.87       576
   macro avg       0.87      0.87      0.87       576
weighted avg       0.87      0.87      0.87       576

[[247  41]
 [ 36 252]]


In [12]:
# Export best ResNet18 model as TorchScript for web use
resnet.load_state_dict(torch.load('best_signature_resnet18.pth', map_location=device))
resnet.eval()
example_input = torch.randn(1, 3, 128, 256).to(device)
traced_resnet = torch.jit.trace(resnet, example_input)
traced_resnet.save('signature_resnet18_script.pt')
print('TorchScript ResNet18 model saved as signature_resnet18_script.pt')

TorchScript ResNet18 model saved as signature_resnet18_script.pt


## 10. Universal Data Pipeline & New Architectures
We now ensure all images are loaded robustly (regardless of name, size, or color), and add a Vision Transformer (ViT) model for further benchmarking.

In [13]:
# Universal transform: robust to size, color, and naming
from torchvision import transforms as T
universal_transform = T.Compose([
    T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_ds_uni = datasets.ImageFolder(train_dir, transform=universal_transform)
val_ds_uni = datasets.ImageFolder(val_dir, transform=universal_transform)
train_loader_uni = DataLoader(train_ds_uni, batch_size=16, shuffle=True)
val_loader_uni = DataLoader(val_ds_uni, batch_size=16)
class_names = train_ds_uni.classes

In [14]:
# Vision Transformer (ViT) from torchvision
try:
    from torchvision.models import vit_b_16, ViT_B_16_Weights
    vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
    for param in vit.parameters():
        param.requires_grad = False
    vit.heads.head = nn.Linear(vit.heads.head.in_features, 2)
    vit = vit.to(device)
    criterion_vit = nn.CrossEntropyLoss()
    optimizer_vit = optim.Adam(vit.heads.head.parameters(), lr=1e-3)
    best_vit_model = None
    best_vit_val_acc = 0.0
    patience = 3
    patience_counter = 0
    num_epochs = 10
    for epoch in range(num_epochs):
        vit.train()
        for imgs, labels in train_loader_uni:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer_vit.zero_grad()
            out = vit(imgs)
            loss = criterion_vit(out, labels)
            loss.backward()
            optimizer_vit.step()
        # Validation
        vit.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader_uni:
                imgs, labels = imgs.to(device), labels.to(device)
                out = vit(imgs)
                preds = out.argmax(1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        print(f"[ViT] Epoch {epoch+1}: Val Acc={val_acc:.4f}")
        if val_acc > best_vit_val_acc:
            best_vit_val_acc = val_acc
            best_vit_model = vit.state_dict()
            patience_counter = 0
            print("  Saved new best ViT model.")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("  Early stopping triggered.")
                break
    # Evaluate best ViT model
    vit.load_state_dict(best_vit_model)
    vit.eval()
    all_preds_vit, all_labels_vit = [], []
    with torch.no_grad():
        for imgs, labels in val_loader_uni:
            imgs = imgs.to(device)
            out = vit(imgs)
            preds = out.argmax(1).cpu().numpy()
            all_preds_vit.extend(preds)
            all_labels_vit.extend(labels.numpy())
    print(classification_report(all_labels_vit, all_preds_vit, target_names=class_names))
    print(confusion_matrix(all_labels_vit, all_preds_vit))
except Exception as e:
    print('ViT not available:', e)

[ViT] Epoch 1: Val Acc=0.9045
  Saved new best ViT model.
[ViT] Epoch 2: Val Acc=0.9201
  Saved new best ViT model.
[ViT] Epoch 3: Val Acc=0.9184
[ViT] Epoch 4: Val Acc=0.9167
[ViT] Epoch 5: Val Acc=0.9288
  Saved new best ViT model.
[ViT] Epoch 6: Val Acc=0.9219
[ViT] Epoch 7: Val Acc=0.9236
[ViT] Epoch 8: Val Acc=0.9306
  Saved new best ViT model.
[ViT] Epoch 9: Val Acc=0.9375
  Saved new best ViT model.
[ViT] Epoch 10: Val Acc=0.9253
              precision    recall  f1-score   support

        fake       0.96      0.89      0.92       288
        real       0.90      0.96      0.93       288

    accuracy                           0.93       576
   macro avg       0.93      0.93      0.93       576
weighted avg       0.93      0.93      0.93       576

[[256  32]
 [ 11 277]]


## 11. Model Results Summary
Below, we summarize the validation accuracy and F1-score for all tested architectures. Use this to select the best model for deployment.

In [15]:
%pip install pandas

# Collect and display results for all models
from sklearn.metrics import f1_score
import pandas as pd

model_scores = {}
# CNN
if 'all_labels' in globals() and 'all_preds' in globals():
    f1_cnn = f1_score(all_labels, all_preds, average='weighted')
    model_scores['CNN'] = f1_cnn
# ResNet
if 'all_labels_resnet' in globals() and 'all_preds_resnet' in globals():
    f1_resnet = f1_score(all_labels_resnet, all_preds_resnet, average='weighted')
    model_scores['ResNet18'] = f1_resnet
# VGG
if 'all_labels_vgg' in globals() and 'all_preds_vgg' in globals():
    f1_vgg = f1_score(all_labels_vgg, all_preds_vgg, average='weighted')
    model_scores['VGG16'] = f1_vgg
# DenseNet
if 'all_labels_dense' in globals() and 'all_preds_dense' in globals():
    f1_dense = f1_score(all_labels_dense, all_preds_dense, average='weighted')
    model_scores['DenseNet'] = f1_dense
# MobileNet
if 'all_labels_mobile' in globals() and 'all_preds_mobile' in globals():
    f1_mobile = f1_score(all_labels_mobile, all_preds_mobile, average='weighted')
    model_scores['MobileNetV2'] = f1_mobile
# EfficientNet
if 'all_labels_eff' in globals() and 'all_preds_eff' in globals():
    f1_eff = f1_score(all_labels_eff, all_preds_eff, average='weighted')
    model_scores['EfficientNet'] = f1_eff
# ViT
if 'all_labels_vit' in globals() and 'all_preds_vit' in globals():
    f1_vit = f1_score(all_labels_vit, all_preds_vit, average='weighted')
    model_scores['ViT'] = f1_vit

if model_scores:
    df = pd.DataFrame(list(model_scores.items()), columns=['Model', 'F1 Score'])
    display(df.sort_values('F1 Score', ascending=False))
else:
    print('No model results available yet. Run all model cells.')

Note: you may need to restart the kernel to use updated packages.


,Model,F1 Score
2,ViT,0.925248
0,CNN,0.892340
1,ResNet18,0.866309


In [16]:
# 12. Save Only the Best Model
import torch
import os

# Map model names to candidate variable names and output file paths
model_objects = {
    'CNN': (['model'], 'best_signature_cnn.pth'),
    'ResNet18': (['resnet', 'resnet_model'], 'best_signature_resnet18.pth'),
    'VGG16': (['vgg', 'vgg_model'], 'best_signature_vgg16.pth'),
    'DenseNet': (['dense', 'dense_model'], 'best_signature_densenet.pth'),
    'MobileNetV2': (['mobile', 'mobile_model'], 'best_signature_mobilenetv2.pth'),
    'EfficientNet': (['eff', 'eff_model'], 'best_signature_efficientnet.pth'),
    'ViT': (['vit', 'vit_model'], 'best_signature_vit.pth'),
}

# Optional fallback state_dict variables if model object is not present
state_dict_fallbacks = {
    'ResNet18': 'best_resnet_model',
    'ViT': 'best_vit_model',
}

if model_scores:
    best_model_name = max(model_scores, key=model_scores.get)
    print(f"Best model: {best_model_name} (F1: {model_scores[best_model_name]:.4f})")
    candidate_vars, model_path = model_objects[best_model_name]

    # Remove all other model files
    for name, (_, path) in model_objects.items():
        if name != best_model_name and os.path.exists(path):
            os.remove(path)

    # Resolve model object by trying candidate variable names
    best_model = None
    for var_name in candidate_vars:
        if var_name in globals():
            best_model = globals()[var_name]
            break

    # Save model
    if best_model is not None:
        torch.save(best_model.state_dict(), model_path)
        print(f"Saved only the best model to {model_path}")
    elif best_model_name in state_dict_fallbacks and state_dict_fallbacks[best_model_name] in globals():
        torch.save(globals()[state_dict_fallbacks[best_model_name]], model_path)
        print(f"Saved only the best model state_dict to {model_path}")
    else:
        print(f"Could not find model variable for {best_model_name}. Available globals do not include expected names: {candidate_vars}")
else:
    print("No model scores available. Run all model cells first.")


Best model: ViT (F1: 0.9252)
Saved only the best model to best_signature_vit.pth
